In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] Failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 16. Positional Encoding — Injecting Order Information

> **Learning Goals**
> - Explain why a transformer has no inherent order information
> - Understand the mathematics of Sinusoidal, Learned, ALiBi, and RoPE encodings
> - Know why the RoPE rotation matrix is the modern LLM standard

## 16.1 The Order Information Problem

Attention is **permutation equivariant**: if the input order changes, the output changes in the same way → order information is lost.

"I ate an apple" vs "The apple ate me" — attention alone cannot distinguish the order.

Solution: add position information to embeddings. Several methods have been proposed.

## 16.2 Sinusoidal Positional Encoding (Original Transformer)

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

- $pos$: token position
- $i$: dimension index
- The frequency changes geometrically as $10000^{-2i/d}$ → encodes position at multiple scales

Simply add it to the embedding: $\mathbf{x} = \mathbf{e} + PE_{pos}$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sinusoidal_pe(seq_len, d_model):
    """Sinusoidal Positional Encoding."""
    pe = np.zeros((seq_len, d_model))
    pos = np.arange(seq_len)[:, None]
    div = np.exp(np.arange(0, d_model, 2) * (-np.log(10000) / d_model))
    pe[:, 0::2] = np.sin(pos * div)
    pe[:, 1::2] = np.cos(pos * div)
    return pe

pe = sinusoidal_pe(100, 64)
print(f"PE shape: {pe.shape}")
print(f"PE[0, :8]: {pe[0, :8].round(4)}")
print(f"PE[1, :8]: {pe[1, :8].round(4)}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im = axes[0].imshow(pe, cmap='RdBu', aspect='auto')
plt.colorbar(im, ax=axes[0])
axes[0].set_xlabel('Embedding Dimension')
axes[0].set_ylabel('Position')
axes[0].set_title('Sinusoidal Positional Encoding')

# Values for selected dimensions
for i in [0, 4, 16, 32]:
    axes[1].plot(pe[:, i], label=f'dim {i}', linewidth=1.5)
axes[1].set_xlabel('Position')
axes[1].set_ylabel('PE Value')
axes[1].set_title('PE pattern by dimension')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch16_sinusoidal_pe.png', dpi=100, bbox_inches='tight')
plt.show()


## 16.3 Learned Positional Embedding

BERT and GPT-2 use trainable positional embeddings $\mathbf{E}_{pos} \in \mathbb{R}^{T_{\max} \times d}$.

Advantage: learns an optimal positional representation from data.
Disadvantage: cannot be used beyond the training length $T_{\max}$ (no extrapolation).


In [ ]:
# Learned PE (PyTorch)
import torch
import torch.nn as nn

max_len, d = 512, 64
learned_pe = nn.Embedding(max_len, d)

# Look up embeddings by position index
positions = torch.arange(10).unsqueeze(0)  # (1, 10)
pe_emb = learned_pe(positions)  # (1, 10, d)
print(f"Learned PE: {pe_emb.shape}")
print(f"Initial values are random and optimized during training")


## 16.4 ALiBi — Extrapolatable Positional Encoding

**ALiBi** (Press et al., 2022) does not add positional embeddings. Instead, it adds a bias to the attention scores:

$$\mathrm{softmax}\left(QK^\top + m \cdot (i - j)\right)$$

- $m$: per-head slope (decreases geometrically)
- $(i - j)$: relative distance (negative for past tokens)
- More distant tokens receive lower scores

**Advantage**: can extrapolate to sequences longer than the training length.


In [ ]:
# ALiBi text
def alibi_bias(n_heads, seq_len):
    """ALiBi Bias Matrix: (n_heads, T, T)."""
    # Per-head slope: 2^(-8 * h / n_heads), h=1..n_heads
    slopes = 1.0 / (2 ** (8 * torch.arange(1, n_heads + 1).float() / n_heads))  # (n_heads,)
    # Distance Matrix: (T, T), relative[i, j] = j - i (positive for future, negative for past)
    positions = torch.arange(seq_len)
    relative = positions[None, :] - positions[:, None]  # (T, T)
    # broadcast: (n_heads, 1, 1) * (1, T, T) -> (n_heads, T, T)
    bias = slopes[:, None, None] * relative[None, :, :]
    return bias

n_heads, seq_len = 8, 16
bias = alibi_bias(n_heads, seq_len)
print(f"ALiBi bias: {bias.shape}")

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for h in range(n_heads):
    ax = axes[h // 4, h % 4]
    im = ax.imshow(bias[h].numpy(), cmap='RdBu_r')
    ax.set_title(f'Head {h} (slope={1/(2**(8/n_heads))**(h+1):.4f})')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig('../figures/ch16_alibi.png', dpi=100, bbox_inches='tight')
plt.show()
print("Each head applies a different slope so scores decrease with distance.")


## 16.5 RoPE (Rotary Position Embedding) — LLaMA Standard

**RoPE** (Su et al., 2021) rotates queries and keys at position $m$:

$$\mathrm{RoPE}(\mathbf{x}, m) = R_m \mathbf{x}$$

Here $R_m$ is a block-diagonal rotation matrix:

$$R_m = \mathrm{diag}\left(R(m\theta_1), R(m\theta_2), \ldots, R(m\theta_{d/2})\right)$$

$$R(\theta) = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}, \quad \theta_i = 10000^{-2i/d}$$

**Key property**:
$$\langle \mathrm{RoPE}(\mathbf{q}, m), \mathrm{RoPE}(\mathbf{k}, n) \rangle = \langle \mathbf{q}, \mathbf{k} \rangle \text{ (after rotation)}$$

→ The dot product depends only on the **relative distance $m - n$**. This encodes relative position, not absolute position.


In [ ]:
# RoPE text
def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, positions, theta_base=10000.0):
    """x: (B, H, T, d), positions: (T,)."""
    d = x.shape[-1]
    # theta_i = 1 / theta_base^(2i/d)
    freqs = 1.0 / (theta_base ** (torch.arange(0, d, 2).float() / d))  # (d/2,)
    # angles: (T, d/2)
    angles = positions[:, None] * freqs[None, :]
    # cos, sin: (T, d) — text Dimension Iteration
    cos = torch.cos(angles).repeat_interleave(2, dim=-1)  # (T, d)
    sin = torch.sin(angles).repeat_interleave(2, dim=-1)  # (T, d)
    # (1, 1, T, d)
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    return x * cos + rotate_half(x) * sin

# Test
d = 8
T = 4
x = torch.randn(1, 1, T, d)
positions = torch.arange(T)
x_rope = apply_rope(x, positions)
print(f"text: {x.shape}")
print(f"RoPE Application text: {x_rope.shape}")

# Verify the key property: depends only on relative distance
# q at pos 0, k at pos 1 vs q at pos 1, k at pos 2 (same distance 1)
q1 = torch.randn(1, 1, 1, d)
k1 = torch.randn(1, 1, 1, d)
pos_q1 = torch.tensor([0])
pos_k1 = torch.tensor([1])
pos_q2 = torch.tensor([1])
pos_k2 = torch.tensor([2])

dot1 = (apply_rope(q1, pos_q1) * apply_rope(k1, pos_k1)).sum()
dot2 = (apply_rope(q1, pos_q2) * apply_rope(k1, pos_k2)).sum()
print(f"\nRoPE text Distance text Validation:")
print(f"  (q@0, k@1) Inner Product = {dot1.item():.6f}")
print(f"  (q@1, k@2) Inner Product = {dot2.item():.6f}")
print(f"  Difference: {abs(dot1 - dot2).item():.2e} (≈0, same relative distance)")


## 16.6 RoPE Visualization

Visualize how RoPE works in 2D.


In [ ]:
# 2D RoPE Visualization
fig, ax = plt.subplots(figsize=(8, 8))
d = 2
theta = 1.0  # text

# text Vector
v = np.array([2.0, 1.0])
ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1,
          color='blue', linewidth=3, label='Original')

# Rotate at positions 1, 2, 3, and 4
for m in [1, 2, 3, 4]:
    angle = m * theta
    R = np.array([[np.cos(angle), -np.sin(angle)],
                  [np.sin(angle),  np.cos(angle)]])
    v_rot = R @ v
    ax.quiver(0, 0, v_rot[0], v_rot[1], angles='xy', scale_units='xy', scale=1,
              color='red', alpha=0.5 + 0.1 * m, linewidth=2,
              label=f'pos={m} (Rotation {np.degrees(angle):.0f}°)')

ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
ax.set_aspect('equal')
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title('RoPE: rotate a vector at position m by m·θ')
plt.tight_layout()
plt.savefig('../figures/ch16_rope_rotation.png', dpi=100, bbox_inches='tight')
plt.show()
print("=> The farther the position, the larger the rotation. The inner product between two vectors depends only on the difference in rotation angle (=relative distance).")


## 16.7 Long-Context Extension

Limitation of RoPE: performance degrades beyond the training length. Solutions:

- **Position Interpolation (PI)**: scale positions ($m \to m \cdot L_{train}/L_{test}$)
- **NTK-aware**: adjust frequencies nonlinearly
- **YaRN**: NTK improvement + scaling

Used by LLaMA-3, Qwen, and other models to support long contexts (128K+).


## 16.8 Key Takeaways

| Method | Extrapolation | Used By |
|---|---|---|
| Sinusoidal | Yes (theoretical) | Original Transformer |
| Learned | No | BERT, GPT-2 |
| ALiBi | Yes | BLOOM |
| RoPE | Partial (requires scaling) | LLaMA, Qwen, Mistral |

## Exercises

1. Compute and compare the frequencies of dimension 0 and dimension 32 in sinusoidal PE.
2. Train learned PE for 100 positions, then visualize the difference between positions 0-50 and 51-99.
3. Apply ALiBi's per-head slope formula $m_h = 2^{-8h/H}$ and simulate its effect on long sequences.
4. Verify RoPE's "depends only on relative distance" property for several position pairs.
5. Explain the extrapolation effect of applying Position Interpolation $m \to m \cdot L_{train}/L_{test}$ to RoPE.

> Solutions: `solutions/ch16_solutions.ipynb`
